# 06. PELT Change-Point Detection

This notebook adds **PELT (Pruned Exact Linear Time)** change-point detection to the temporal Product × Aspect series.

## Purpose

EWMA answers:

> **What is the recent direction?**  
> Improving / Stable / Declining

PELT answers:

> **When did a meaningful structural shift occur?**

For this PP1 development version, PELT is applied to the **smoothed sentiment series** for each Product × Aspect pair.

> The penalty, minimum segment size, and matching tolerance used here are development settings only.  
> They must later be tuned and validated using real Component 1 output.


In [1]:
!pip -q install ruptures


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ruptures as rpt
from pathlib import Path

WORKSPACE = Path.cwd()
if WORKSPACE.name == "notebooks":
    COMPONENT_ROOT = WORKSPACE.parent
elif (WORKSPACE / "data" / "processed").exists():
    COMPONENT_ROOT = WORKSPACE
else:
    COMPONENT_ROOT = WORKSPACE / "component_3"

INPUT = COMPONENT_ROOT / "data" / "processed" / "paq_with_ewma_trend.csv"
GROUND_TRUTH = COMPONENT_ROOT / "data" / "processed" / "dummy_c1_ground_truth.csv"

OUTPUT_DIR = COMPONENT_ROOT / "data" / "processed"
FIGURE_DIR = COMPONENT_ROOT / "outputs" / "figures"
TABLE_DIR = COMPONENT_ROOT / "outputs" / "tables"

for p in [OUTPUT_DIR, FIGURE_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

OUTPUT = OUTPUT_DIR / "paq_with_trend_changepoint.csv"

# Temporary PP1 development settings
DEV_PENALTY = 0.08
DEV_MIN_SIZE = 2
DEV_MATCH_TOLERANCE_QUARTERS = 1


## 1. Load the EWMA output


In [3]:
df = pd.read_csv(INPUT)

required = [
    "parent_asin",
    "aspect",
    "quarter",
    "smoothed_sentiment",
    "ewma_sentiment",
    "trend"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["quarter_period"] = pd.PeriodIndex(
    df["quarter"],
    freq="Q"
)

df = (
    df
    .sort_values(
        ["parent_asin", "aspect", "quarter_period"]
    )
    .reset_index(drop=True)
)

print("Rows:", len(df))
print("Products:", df["parent_asin"].nunique())
print(
    "Product-aspect series:",
    df[["parent_asin", "aspect"]].drop_duplicates().shape[0]
)

display(df.head())


Rows: 550
Products: 10
Product-aspect series: 50


,parent_asin,aspect,quarter,review_count,mean_sentiment,median_sentiment,positive_rate,neutral_rate,negative_rate,avg_confidence,...,ci_upper,ci_width,temporal_confidence_dev,confidence_label_dev,ewma_sentiment,ewma_change,trend_signal,trend,trend_evidence_flag_dev,quarter_period
0,B00FFJ0HUE,battery,2021Q1,3,0.1326,-0.1452,0.3333,0.6667,0.0,0.7348,...,0.2712,0.3387,0.6196,Medium,0.0497,NaN,NaN,Stable,Low Support,2021Q1
1,B00FFJ0HUE,battery,2021Q2,3,-0.0371,-0.0177,0.0000,1.0000,0.0,0.7411,...,0.1140,0.1044,0.6567,Medium,0.0574,0.0077,0.0077,Stable,Low Support,2021Q2
2,B00FFJ0HUE,battery,2021Q3,5,0.4642,0.4297,0.8000,0.2000,0.0,0.7929,...,0.3636,0.2242,0.7042,Medium,0.1369,0.0794,0.0436,Stable,Adequate,2021Q3
3,B00FFJ0HUE,battery,2021Q4,5,0.2114,0.1513,0.4000,0.6000,0.0,0.7260,...,0.2866,0.1503,0.6953,Medium,0.1617,0.0249,0.0521,Improving,Adequate,2021Q4
4,B00FFJ0HUE,battery,2022Q1,2,0.0833,0.0833,0.5000,0.0000,0.5,0.8004,...,0.3072,0.2915,0.6107,Medium,0.1616,-0.0001,0.0124,Stable,Low Support,2022Q1


## 2. PELT helper function

For each Product × Aspect series:

1. sort quarters chronologically;
2. use `smoothed_sentiment` as the signal;
3. fit PELT with an L2 cost;
4. remove the final endpoint returned by the library;
5. convert break positions into quarter labels.

If a series is too short, no change point is returned.


In [4]:
def detect_pelt_change_points(
    group,
    penalty=DEV_PENALTY,
    min_size=DEV_MIN_SIZE
):
    group = group.sort_values("quarter_period").copy()

    signal = (
        group["smoothed_sentiment"]
        .astype(float)
        .to_numpy()
        .reshape(-1, 1)
    )

    n = len(signal)

    if n < (2 * min_size + 1):
        return []

    model = rpt.Pelt(
        model="l2",
        min_size=min_size,
        jump=1
    ).fit(signal)

    break_positions = model.predict(
        pen=penalty
    )

    # ruptures includes the end of the signal as the final breakpoint.
    internal_breaks = [
        b for b in break_positions
        if b < n
    ]

    quarters = group["quarter"].tolist()

    detected = []

    for b in internal_breaks:
        # b is the first index after the previous segment.
        # Report the quarter at which the new segment starts.
        if 0 <= b < len(quarters):
            detected.append(quarters[b])

    return detected


## 3. Detect change points for all Product × Aspect series


In [5]:
change_rows = []

for (product, aspect), group in df.groupby(
    ["parent_asin", "aspect"]
):
    detected = detect_pelt_change_points(group)

    change_rows.append({
        "parent_asin": product,
        "aspect": aspect,
        "n_quarters": len(group),
        "n_change_points": len(detected),
        "detected_change_points": "|".join(detected),
        "latest_change_point": (
            detected[-1] if detected else ""
        )
    })

change_summary = pd.DataFrame(change_rows)

change_summary.to_csv(
    TABLE_DIR / "pelt_change_points.csv",
    index=False
)

display(
    change_summary
    .sort_values(
        ["n_change_points", "parent_asin", "aspect"],
        ascending=[False, True, True]
    )
)


,parent_asin,aspect,n_quarters,n_change_points,detected_change_points,latest_change_point
3,B00FFJ0HUE,performance,11,1,2021Q3,2021Q3
7,B00KB7O49K,display,11,1,2021Q4,2021Q4
17,B00NOSVK54,display,11,1,2022Q2,2022Q2
29,B019BX7F86,thermal,11,1,2023Q1,2023Q1
34,B01K1IO3QW,thermal,11,1,2022Q4,2022Q4
35,B01MPXRXR0,battery,11,1,2022Q4,2022Q4
45,B0C2ZMJW53,battery,11,1,2022Q4,2022Q4
0,B00FFJ0HUE,battery,11,0,,
1,B00FFJ0HUE,display,11,0,,
2,B00FFJ0HUE,keyboard,11,0,,


## 4. Attach the latest detected change point to the PAQ output


In [6]:
df = df.merge(
    change_summary[[
        "parent_asin",
        "aspect",
        "n_change_points",
        "detected_change_points",
        "latest_change_point"
    ]],
    on=["parent_asin", "aspect"],
    how="left"
)

display(
    df[[
        "parent_asin",
        "aspect",
        "quarter",
        "smoothed_sentiment",
        "ewma_sentiment",
        "trend",
        "n_change_points",
        "latest_change_point"
    ]].head(30)
)


,parent_asin,aspect,quarter,smoothed_sentiment,ewma_sentiment,trend,n_change_points,latest_change_point
0,B00FFJ0HUE,battery,2021Q1,0.0497,0.0497,Stable,0,
1,B00FFJ0HUE,battery,2021Q2,0.0690,0.0574,Stable,0,
2,B00FFJ0HUE,battery,2021Q3,0.2560,0.1369,Stable,0,
3,B00FFJ0HUE,battery,2021Q4,0.1990,0.1617,Improving,0,
4,B00FFJ0HUE,battery,2022Q1,0.1615,0.1616,Stable,0,
5,B00FFJ0HUE,battery,2022Q2,0.2876,0.2120,Stable,0,
6,B00FFJ0HUE,battery,2022Q3,0.4333,0.3005,Improving,0,
7,B00FFJ0HUE,battery,2022Q4,0.2768,0.2910,Stable,0,
8,B00FFJ0HUE,battery,2023Q1,0.3133,0.2999,Stable,0,
9,B00FFJ0HUE,battery,2023Q2,0.3014,0.3005,Stable,0,


## 5. Validate PELT on the synthetic ground truth

The dummy development dataset contains intentionally planted changes.

This gives a useful PP1 test because the approximate true change quarter is known.

A detected change is counted as a match if it falls within ±1 quarter of the planted change.

> This is a **synthetic development test**, not final research evaluation.


In [7]:
def quarter_distance(q1, q2):
    p1 = pd.Period(q1, freq="Q")
    p2 = pd.Period(q2, freq="Q")
    return abs(p1.ordinal - p2.ordinal)

if GROUND_TRUTH.exists():
    truth = pd.read_csv(GROUND_TRUTH)

    eval_rows = []

    for _, truth_row in truth.iterrows():
        found = change_summary[
            (change_summary["parent_asin"] == truth_row["parent_asin"])
            & (change_summary["aspect"] == truth_row["aspect"])
        ]

        detected_list = []

        if len(found) == 1:
            raw_value = found.iloc[0]["detected_change_points"]

            if isinstance(raw_value, str) and raw_value.strip():
                detected_list = raw_value.split("|")

        distances = [
            quarter_distance(
                q,
                truth_row["change_quarter"]
            )
            for q in detected_list
        ]

        if distances:
            best_distance = min(distances)
            matched = (
                best_distance
                <= DEV_MATCH_TOLERANCE_QUARTERS
            )
        else:
            best_distance = np.nan
            matched = False

        eval_rows.append({
            "parent_asin": truth_row["parent_asin"],
            "aspect": truth_row["aspect"],
            "pattern": truth_row["pattern"],
            "true_change_quarter": truth_row["change_quarter"],
            "detected_change_points": "|".join(detected_list),
            "best_distance_quarters": best_distance,
            "matched_within_tolerance": matched
        })

    synthetic_eval = pd.DataFrame(eval_rows)

    synthetic_eval.to_csv(
        TABLE_DIR / "pelt_synthetic_validation.csv",
        index=False
    )

    display(synthetic_eval)

    recall = synthetic_eval[
        "matched_within_tolerance"
    ].mean()

    print(
        "Synthetic planted-change recall:",
        round(recall, 3)
    )

else:
    print(
        "Synthetic ground truth file not found. "
        "Skipping development validation."
    )


Synthetic ground truth file not found. Skipping development validation.


## 6. Plot one planted-change example

If the synthetic ground-truth file is available, the first planted case is plotted.

The figure shows:
- smoothed sentiment;
- EWMA;
- planted change quarter;
- PELT-detected change quarter(s).


In [8]:
if GROUND_TRUTH.exists():
    truth = pd.read_csv(GROUND_TRUTH)

    example_truth = truth.iloc[0]

    example_product = example_truth["parent_asin"]
    example_aspect = example_truth["aspect"]
    true_change = example_truth["change_quarter"]

    example = df[
        (df["parent_asin"] == example_product)
        & (df["aspect"] == example_aspect)
    ].copy()

    example = example.sort_values("quarter_period")

    detected_raw = change_summary[
        (change_summary["parent_asin"] == example_product)
        & (change_summary["aspect"] == example_aspect)
    ].iloc[0]["detected_change_points"]

    detected = (
        detected_raw.split("|")
        if isinstance(detected_raw, str)
        and detected_raw.strip()
        else []
    )

    plt.figure(figsize=(10, 5))

    plt.plot(
        example["quarter"],
        example["smoothed_sentiment"],
        marker="o",
        label="Smoothed sentiment"
    )

    plt.plot(
        example["quarter"],
        example["ewma_sentiment"],
        marker="o",
        label="EWMA"
    )

    plt.axvline(
        true_change,
        linestyle="--",
        linewidth=1.5,
        label=f"Planted change: {true_change}"
    )

    for i, q in enumerate(detected):
        plt.axvline(
            q,
            linestyle=":",
            linewidth=1.5,
            label=(
                f"PELT detected: {q}"
                if i == 0
                else None
            )
        )

    plt.axhline(
        0,
        linestyle="--",
        linewidth=1
    )

    plt.title(
        f"PELT Change-Point Detection\n"
        f"{example_product} — {example_aspect}"
    )

    plt.xlabel("Quarter")
    plt.ylabel("Sentiment score")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / "pelt_change_point_example.png",
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

    print("Example product:", example_product)
    print("Example aspect:", example_aspect)
    print("True change:", true_change)
    print("Detected:", detected)


## 7. Save the integrated Function 1 output


In [9]:
df = df.drop(
    columns=["quarter_period"]
)

df.to_csv(
    OUTPUT,
    index=False
)

print("Saved:", OUTPUT)
print("Rows:", len(df))


Saved: c:\Users\victus\Desktop\Repository\J26-DS-349\component_3\data\processed\paq_with_trend_changepoint.csv
Rows: 550
